In [1]:
!pip install chromadb openai pypdf tiktoken pandas rank-bm25

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---- ----------------------------------- 2.4/23.5 MB 12.9 MB/s eta 0:00:02
   ----------- ---------------------------- 6.6/23.5 MB 16.7 MB/s eta 0:00:02
   -------------------- ------------------- 12.1/23.5 MB 20.2 MB/s eta 0:00:01
   ------------------------------ --------- 18.1/23.5 MB 22.2 MB/s eta 0:00:01
   ---------------------------------------  23.3/23.5 MB 23.1 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 22.0 MB/s  0:00:01
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 22.4 MB/s  0:00:00
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 19.0 MB/s  0:00:00
   ---------------------------------------- 0.0/5.0 MB ? eta -:--:--
   ---------------------------------------- 5.0/5.0 MB 25.9 MB/s  0:00:00
   -----------------------

In [41]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from pypdf import PdfReader
import tiktoken

def extract_text_from_pdf(path):
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append({"page_num": i + 1, "text": text})
    return pages

def chunk_text(pages,doc_name,chunk_size = 500,overlap = 50):
    encoding = tiktoken.get_encoding("cl100k_base")
    chunks = []
    for page in pages:
        tokens = encoding.encode(page["text"])
        start = 0
        while start < len(tokens):
            end = min(start + chunk_size, len(tokens))
            chunk_tokens = tokens[start:end]
            chunk_str = encoding.decode(chunk_tokens)
            chunks.append({
                "text": chunk_str,
                "source": doc_name,
                "page": page["page_num"],
                "chunk_id": f"{doc_name}_p{page['page_num']}_c{len(chunks)}"
            })
            start += chunk_size - overlap
    return chunks

def process_documents(folder = "documents"):
    all_chunks = []
    for filename in os.listdir(folder):
        if filename.endswith(".pdf"):
            path = os.path.join(folder,filename)
            pages = extract_text_from_pdf(path)
            chunks = chunk_text(pages,filename)
            all_chunks.extend(chunks)
            print(f"{filename}:{len(chunks)}chunks")
    return all_chunks

In [9]:
chunks = process_documents("documents")
print(f"\nTotal chunks: {len(chunks)}")

apple_10k_2023.pdf:149chunks
apple_10k_2024.pdf:155chunks
apple_10k_2025.pdf:156chunks

Total chunks: 460


In [17]:
from openai import OpenAI

client = OpenAI(api_key = os.environ["OPENAI_API_KEY"])
def embed_texts(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(input = texts, model = model)
    return[item.embedding for item in response.data]

def embed_chunks(chunks, batch_size = 100):
    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        embeddings = embed_texts(texts)
        for chunk, emb in zip (batch,  embeddings):
            chunk["embedding"] = emb
        print(f"Embedded{min(i+batch_size, len(chunks))}/{len(chunks)}chunks")
    return chunks

In [19]:
chunks = embed_chunks(chunks)
print("\nDone. Example embedding(first 5 numbers):")
print(chunks[0]["embedding"][:5])

Embedded100/460chunks
Embedded200/460chunks
Embedded300/460chunks
Embedded400/460chunks
Embedded460/460chunks

Done. Example embedding(first 5 numbers):
[0.031982421875, 0.006244659423828125, 0.00974273681640625, 0.032318115234375, 0.0122528076171875]


In [20]:
!pip install chromadb

In [23]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("apple-10K")

collection.add(
    ids = [c["chunk_id"] for c in chunks],
    embeddings = [c["embedding"] for c in chunks],
    documents = [c["text"] for c in chunks],
    metadatas = [{"source":c["source"],"page":c["page"]} for c in chunks]
)
print(f"stored{collection.count()} chunks in the vector database")

stored460 chunks in the vector database


In [25]:
def search(question, n_results = 5):
    query_embedding = embed_texts([question])[0]
    results = collection.query(
        query_embeddings = [query_embedding],
        n_results = n_results
    )

    retrieved = []
    for doc, meta in zip(results["documents"][0],results["metadatas"][0]):
        retrieved.append({
            "text":doc,
            "source": meta["source"],
            "page": meta["page"]
        })
    return retrieved

In [26]:
results = search("what was Apple's total net sales?")

for i, r in enumerate(results):
    print(f"--- Result {i+1} (Source:{r['source']}, page:{r['page']})---")
    print(r["text"][:300])
    print()

--- Result 1 (Source:apple_10k_2024.pdf, page:33)---
Products and Services Performance
The following table shows net sales by category for 2024, 2023 and 2022 (dollars in millions):
2024 Change 2023 Change 2022
iPhone $ 201,183 — % $ 200,583 (2)% $ 205,489 
Mac 29,984 2 % 29,357 (27)% 40,177 
iPad 26,694 (6)% 28,300 (3)% 29,292 
Wearables, Home and Ac

--- Result 2 (Source:apple_10k_2023.pdf, page:61)---
The U.S. and China were the only countries that accounted for more than 10% of the Company’s net sales in 2023, 2022 and 2021. Net
sales for 2023, 2022 and 2021 and long-lived assets as of September 30, 2023 and September 24, 2022 were as follows (in millions):
2023 2022 2021
Net sales:
U.S. $ 138,5

--- Result 3 (Source:apple_10k_2023.pdf, page:33)---
Products and Services Performance
The following table shows net sales by category for 2023, 2022 and 2021 (dollars in millions):
2023 Change 2022 Change 2021
Net sales by category:
iPhone $ 200,583 (2)% $ 205,489 7 % $ 191,973 
Mac 29,35

In [31]:
def generate_answer(question, n_results=5):
    retrieved = search(question, n_results =n_results)

    context_blocks = []
    for r in retrieved:
        context_blocks.append(f"[Source:{r['source']}, page{r['page']}]\n{r['text']}")
    context = "\n\n---\n\n".join(context_blocks)

    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY
the context below. If the answer isn't in the context, say so clearly.

for every clain, cite the source and page in brackets, e.g.[apple_10k_2024.pdf,p.33].

context:
{context}

Question: {question}

Answer:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content, retrieved

In [32]:
answer, sources = generate_answer("How did Apple's iPhone revenue change from 2023 to 2024?")
print(answer)

Apple's iPhone revenue was relatively flat during 2024 compared to 2023, with net sales of $201,183 million in 2024 compared to $200,583 million in 2023, indicating no significant change in percentage terms [apple_10k_2024.pdf, p.33].


In [33]:
test_set = [
    {"question": "What was Apple's total net sales in fiscal 2024?", "expected_source": "apple_10k_2024.pdf"},
    {"question": "How much did Apple spend on research and development in 2023?", "expected_source": "apple_10k_2025.pdf"},
    {"question": "What percentage of Apple's sales came from the Americas segment in 2025", "expected_source": "apple_10k_2025.pdf"},
    {"question": "What are Apple's main risk factors related to supply chain?", "expected_source":"apple_10k_2024.pdf"},
    {"question": "How much cash and marketable securities did Apple hold in 2025?", "expected_source": "apple_10k_2025.pdf"},
]

In [34]:
def evaluate_retrieval(test_set, n_results = 5):
    hits = 0
    for case in test_set:
        results = search(case["question"], n_results=n_results)
        found = any(r["source"] == case["expected_source"] for r in results)
        status = "✓" if found else "x"
        print(f"{status}{case['question']}")
        if not found:
            print(f"  Expected:{case['expected_source']}, Got: {[r['source'] for r in results]}")
        hits += found

    recall = hits/len(test_set)
    print(f"\nRetrieval Recall@{n_results}: {recall:.0%} ({hits}/{len(test_set)})")
    return recall

evaluate_retrieval(test_set)

✓What was Apple's total net sales in fiscal 2024?
✓How much did Apple spend on research and development in 2023?
✓What percentage of Apple's sales came from the Americas segment in 2025
✓What are Apple's main risk factors related to supply chain?
✓How much cash and marketable securities did Apple hold in 2025?

Retrieval Recall@5: 100% (5/5)


1.0

In [37]:
import chromadb
check_client = chromadb.PersistentClient(path="./chroma_db")
print(check_client.list_collections())

[Collection(name=apple-10K)]


In [38]:
with open(".gitignore", "w") as f:
    f.write(".env\nchroma_db/\ndocuments/*.pdf\n.ipynb_checkpoints/\n__pycache__/\n")

In [39]:
with open(".gitignore", "r") as f:
    print(f.read())

.env
chroma_db/
documents/*.pdf
.ipynb_checkpoints/
__pycache__/



In [2]:
with open("requirements.txt", "w") as f:
    f.write("""streamlit
openai
chromadb
pypdf
tiktoken
pandas
python-dotenv
rank-bm25
sentence-transformers
""")

In [43]:
with open(".gitignore", "w") as f:
    f.write(".env\ndocuments/*.pdf\n.ipynb_checkpoints/\n__pycache__/\n")

In [1]:
!pip install sentence-transformers

   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 13.5 MB/s  0:00:00
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   ----------------- ---------------------- 5.0/11.6 MB 24.1 MB/s eta 0:00:01
   ----------------------------------- ---- 10.2/11.6 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 22.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 24.6 MB/s  0:00:00
   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   - -------------------------------------- 5.8/122.1 MB 30.0 MB/s eta 0:00:04
   ---- ----------------------------------- 12.6/122.1 MB 31.5 MB/s eta 0:00:04
   ------ --------------------------------- 19.7/122.1 MB 31.6 MB/s eta 0:00:04
   -------- ------------------------------- 26.7/122.1 MB 32.0 MB/s eta 0:00:03
   ---------- -

In [3]:
with open("requirements.txt", "r") as f:
    print(f.read())

streamlit
openai
chromadb
pypdf
tiktoken
pandas
python-dotenv
rank-bm25
sentence-transformers



In [4]:
for c in chunks:
    if "risk factor" in c["text"].lower() and c["source"] == "apple_10k_2024.pdf":
        print(c["page"], c["text"][:200])
        print("---")

NameError: name 'chunks' is not defined

In [5]:
import chromadb
check_client = chromadb.PersistentClient(path="./chroma_db")
check_collection = check_client.get_collection("apple-10K")

all_data = check_collection.get(include=["documents", "metadatas"])

for doc, meta in zip(all_data["documents"], all_data["metadatas"]):
    if "risk factor" in doc.lower() and meta["source"] == "apple_10k_2024.pdf":
        print(meta["page"], doc[:200])
        print("---")

3 Apple Inc.
Form 10-K
For the Fiscal Year Ended September 28, 2024
TABLE OF CONTENTS
Page
Part I
Item 1. Business 1
Item 1A. Risk Factors 5
Item 1B. Unresolved Staff Comments 17
Item 1C. Cybersecurity 
---
4 This Annual Report on Form 10-K (“Form 10-K”) contains forward-looking statements, within the meaning of the Private Securities
Litigation Reform Act of 1995, that involve risks and uncertainties. Man
---
9 Available Information
The Company’s Annual Reports on Form 10-K, Quarterly Reports on Form 10-Q, Current Reports on Form 8-K, and amendments to
reports filed pursuant to Sections 13(a) and 15(d) of th
---
26  material risks from cybersecurity threats. The Company’s Head of Corporate Information Security leads a dedicated
Information Security team of highly skilled individuals with experience across indust
---
34 Gross Margin
Products and Services gross margin and gross margin percentage for 2024, 2023 and 2022 were as follows (dollars in millions):
2024 2023 2022
Gross margin:


In [6]:
for doc, meta in zip(all_data["documents"], all_data["metadatas"]):
    if meta["source"] == "apple_10k_2024.pdf" and 5 <= meta["page"] <= 17:
        print(f"--- Page {meta['page']} ---")
        print(doc[:200])
        print()

--- Page 5 ---
Services
Advertising
The Company’s advertising services include third-party licensing arrangements and the Company’s own advertising platforms.
AppleCare
The Company offers a portfolio of fee-based se

--- Page 5 ---
 through its retail and
online stores and its direct sales force. The Company also employs a variety of indirect distribution channels, such as third-party cellular
network carriers, wholesalers, reta

--- Page 6 ---
The Company’s ability to compete successfully depends heavily on ensuring the continuing and timely introduction of innovative new
products, services and technologies to the marketplace. The Company d

--- Page 6 ---
 product uses new technologies, initial
capacity constraints may exist until the suppliers’ yields have matured or their manufacturing capacities have increased. The continued
availability of these co

--- Page 6 ---
 may be necessary in the future to seek or renew licenses relating to various aspects of the Company’s
products, proc

In [7]:
with open(".gitignore", "w") as f:
    f.write(".env\ndocuments/*.pdf\n.ipynb_checkpoints/\n__pycache__/\nquery_logs.db\n")